# 可选实验：Cost Function

<figure style="text-align:center;">
  <img src="./images/C1_W1_L3_S2_Lecture_b.png" alt="Cost function 课程图示" style="width:1000px;max-width:100%;height:auto;">
</figure>


## 目标

在本实验中，你将：

- 实现并探索单变量 linear regression 的 cost function（代价函数）。


## 工具

本实验将使用：

- NumPy：常用的科学计算库。
- Matplotlib：常用的数据绘图库。
- 当前目录下 `lab_utils_uni.py` 中的绘图函数。


In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from lab_utils_uni import plt_intuition, plt_stationary, plt_update_onclick, soup_bowl
plt.style.use('./deeplearning.mplstyle')

## 问题描述

我们希望构建一个根据房屋面积预测价格的模型。继续使用上一个实验的两个数据点：面积为 1000 sqft 的房屋售价为 300,000 美元，面积为 2000 sqft 的房屋售价为 500,000 美元。

| 面积（1000 sqft） | 价格（千美元） |
| --- | --- |
| 1 | 300 |
| 2 | 500 |


In [ ]:
x_train = np.array([1.0, 2.0])           #(size in 1000 square feet)
y_train = np.array([300.0, 500.0])           #(price in 1000s of dollars)

## 计算 Cost

这里需要区分 cost 与房屋价格：cost 衡量模型对目标房价的预测效果，而房屋本身的价格称为 price。

单变量模型的 cost 为：

$$
J(w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{w,b}(x^{(i)})-y^{(i)}\right)^2 \tag{1}
$$

其中：

$$
f_{w,b}(x^{(i)})=wx^{(i)}+b \tag{2}
$$

- $f_{w,b}(x^{(i)})$ 是使用参数 $w,b$ 对第 $i$ 个样本的预测。
- $(f_{w,b}(x^{(i)})-y^{(i)})^2$ 是预测值与 target 之差的平方。
- 将全部 $m$ 个样本的 squared error 相加，再除以 $2m$，得到 cost $J(w,b)$。

> 注意：课程视频中的求和 index 通常从 1 到 $m$，而代码中从 0 到 $m-1$。


下面的代码通过遍历所有样本计算 cost。每次循环会：

- 计算预测值 `f_wb`。
- 计算预测值与 target 之差的平方。
- 将该值累加，最后统一除以 $2m$。


In [ ]:
def compute_cost(x, y, w, b): 
    """
    Computes the cost function for linear regression.
    
    Args:
      x (ndarray (m,)): Data, m examples 
      y (ndarray (m,)): target values
      w,b (scalar)    : model parameters  
    
    Returns
        total_cost (float): The cost of using w,b as the parameters for linear regression
               to fit the data points in x and y
    """
    # number of training examples
    m = x.shape[0] 
    
    cost_sum = 0 
    for i in range(m): 
        f_wb = w * x[i] + b   
        cost = (f_wb - y[i]) ** 2  
        cost_sum = cost_sum + cost  
    total_cost = (1 / (2 * m)) * cost_sum  

    return total_cost

## 直观理解 Cost Function


<img align="left" src="./images/C1_W1_Lab02_GoalOfRegression.PNG" alt="Regression 的优化目标" style="width:380px;padding:10px;">

你的目标是找到模型 $f_{w,b}(x)=wx+b$ 的参数 $w,b$，使模型能够根据输入 $x$ 准确预测房价。Cost 衡量模型在训练数据上的预测误差。

根据公式 (1)，如果选择的 $w,b$ 使预测值 $f_{w,b}(x)$ 与 target $y$ 完全一致，那么 $(f_{w,b}(x^{(i)})-y^{(i)})^2$ 为 0，cost 也达到最小值。在这个只有两个数据点的例子中，我们可以做到这一点！

上一个实验已确定 $b=100$ 可以得到最优解，因此先固定 $b=100$，重点观察 $w$。

使用下面的滑块选择使 cost 最小的 $w$。图表更新可能需要几秒钟。


In [ ]:
plt_intuition(x_train,y_train)

观察图表，可以发现：

- 当 $w=200$ 时，cost 最小，与上一个实验的结果一致。
- Cost 公式对预测误差取平方，因此 $w$ 过大或过小时，cost 都会迅速上升。
- 使用使 cost 最小的 `w` 和 `b`，得到的直线恰好经过所有数据点。


## Cost Function 的三维可视化

通过 3D 图或 contour plot（等高线图），可以观察 cost 如何同时随 `w` 和 `b` 变化。

本课程的一些绘图代码较复杂，相关函数已提供在当前目录的 `lab_utils_uni.py` 中。阅读这些代码有助于了解绘图方法，但并不是完成课程的必要条件。


### 更大的数据集

下面观察包含更多数据点的情况。这些点并不都在同一条直线上，这对 cost 有什么影响？还能否找到使 cost 为 0 的 $w$ 和 $b$？


In [ ]:
x_train = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])
y_train = np.array([250, 300, 480,  430,   630, 730,])

在 contour plot 中点击一个点，选择 `w` 和 `b`，尝试得到最低的 cost。可以借助等高线判断方向。图表更新可能需要几秒钟。


In [ ]:
plt.close('all') 
fig, ax, dyn_items = plt_stationary(x_train, y_train)
updater = plt_update_onclick(fig, ax, x_train, y_train, dyn_items)

注意左图中的虚线：它们表示每个训练样本的预测误差，平方后构成该样本对 cost 的贡献。在本例中，$w\approx209$、$b\approx2.4$ 可以得到较低的 cost。由于训练样本不在同一条直线上，最小 cost 不为 0。


### Convex Cost Surface

对于这里的 linear regression，squared error cost function 形成了类似碗状的 convex surface（凸曲面）。使用合适的 learning rate，沿负 gradient 方向更新各个参数，可以到达最小值。

前面的图中，$w$ 和 $b$ 的坐标轴尺度不同，因此不容易看出这种形状。下面是课程中展示的、使用更对称尺度的图：


In [ ]:
soup_bowl()

## 恭喜完成实验！

你已经了解到：

- Cost 衡量预测值与训练数据的匹配程度。
- 最小化 cost 可以求得最优参数 $w,b$。
